# Tối ưu hóa và Làm sạch dữ liệu Shopee

Notebook này thực hiện quy trình chuẩn hóa dữ liệu từ file gốc `Shopee_Products_Master.csv`.

**Cập nhật mới (V3):**
- Sử dụng `utf-8-sig` để **sửa lỗi font tiếng Việt**.
- **Tinh chỉnh dữ liệu History:**
    - Loại bỏ cột `item_name` dư thừa (đã có trong Master).
    - Parse cột `discount` từ dạng chuỗi (ví dụ '45%') sang số thực (0.45).
    - Sắp xếp thứ tự cột hợp lý hơn (ItemId -> Time).

**Mục tiêu:**
1.  **Làm sạch:** Loại bỏ các dữ liệu rác (sản phẩm bị khóa, ẩn).
2.  **Chuẩn hóa:** Chuyển đổi định dạng thời gian, xử lý giá trị thiếu.
3.  **Chuẩn hóa tên:** Loại bỏ rác, tag quảng cáo.
4.  **Tách dữ liệu:** Tạo ra 2 file `Core_Product_Master.csv` và `Price_Stock_History.csv` tối ưu cho AI Model.

In [8]:
import csv
import os
import re
from datetime import datetime

# --- CẤU HÌNH ĐƯỜNG DẪN ---
INPUT_FILE = r"D:\GREENMIND\data_vn_ecommerce\Shopee_Products_Master.csv"
OUTPUT_DIR = r"D:\GREENMIND\data_vn_ecommerce\Optimized"

OUTPUT_MASTER = os.path.join(OUTPUT_DIR, "Core_Product_Master.csv")
OUTPUT_HISTORY = os.path.join(OUTPUT_DIR, "Price_Stock_History.csv")

if not os.path.exists(OUTPUT_DIR):
    os.makedirs(OUTPUT_DIR)


In [9]:
def clean_item_name(name):
    if not name:
        return ""
    # 1. Loại bỏ tag trong ngoặc
    cleaned = re.sub(r'\(.*?\)', '', name)
    cleaned = re.sub(r'\[.*?\]', '', cleaned)
    # 2. Loại bỏ ký tự đặc biệt đầu cuối
    cleaned = cleaned.strip(" -.,|:")
    # 3. Loại bỏ khoảng trắng thừa
    cleaned = " ".join(cleaned.split())
    return cleaned

def parse_discount(discount_str):
    """Chuyển '45%' thành 0.45, '' thành 0.0"""
    if not discount_str:
        return 0.0
    try:
        if '%' in discount_str:
            return float(discount_str.replace('%', '')) / 100
        return float(discount_str)
    except ValueError:
        return 0.0

def process_shopee_data():
    print(f"Đang đọc dữ liệu từ: {INPUT_FILE} ...")
    
    unique_products = {}
    history_rows = []
    
    stats = {
        'total_rows': 0,
        'valid_rows': 0,
        'banned_rows': 0,
    }

    # Header mới cho file History: ItemID là khóa chính, bỏ item_name
    history_headers = ['itemid', 'time_str', 'date', 'price', 'original_price', 
                       'discount', 'stock', 'sold', 'cmt_count', 'liked_count']

    try:
        with open(INPUT_FILE, mode='r', encoding='utf-8') as f_in:
            reader = csv.DictReader(f_in)
            
            for row in reader:
                stats['total_rows'] += 1
                
                # BƯỚC 1: LỌC RÁC
                status = row.get('item_status', '').lower()
                if status in ['banned', 'offensive_hide', 'deleted']:
                    stats['banned_rows'] += 1
                    continue
                
                # BƯỚC 2: CHUẨN HÓA
                try:
                    ts = int(row['time'])
                    dt_object = datetime.fromtimestamp(ts)
                    time_str = dt_object.strftime('%Y-%m-%d %H:%M:%S')
                except:
                    time_str = row['time']
                
                discount_raw = row.get('discount', '').strip()
                discount_val = parse_discount(discount_raw)
                    
                price_before = row.get('price_before_discount', '').strip()
                current_price = row.get('price', '0')
                if not price_before:
                    price_before = current_price
                    
                raw_name = row.get('item_name', '')
                cleaned_name = clean_item_name(raw_name)
                
                # BƯỚC 3: TÁCH DATA (Đã tối ưu)
                # History: Chỉ chứa số liệu biến động
                history_item = {
                    'itemid': row.get('itemid', ''),
                    'time_str': time_str,
                    'date': row.get('date', ''),
                    'price': current_price,
                    'original_price': price_before,
                    'discount': discount_val,  # Đã chuẩn hóa thành số (0.x)
                    'stock': row.get('stock', '0'),
                    'sold': row.get('sold', '0'),
                    'cmt_count': row.get('cmt_count', '0'),
                    'liked_count': row.get('liked_count', '0')
                }
                history_rows.append(history_item)
                stats['valid_rows'] += 1
                
                # Master: Chứa thông tin mô tả sản phẩm
                item_id = row.get('itemid')
                is_latest = (row.get('is_latest', '').lower() == 'true')
                
                if item_id not in unique_products or is_latest:
                    unique_products[item_id] = {
                        'itemid': item_id,
                        'item_name': cleaned_name,
                        'raw_name': raw_name,
                        'current_price': current_price,
                        'total_sold': row.get('sold', '0'),
                        'last_updated': time_str,
                        'status': status
                    }

        print("\n--- KẾT QUẢ XỬ LÝ (TỐI ƯU V3) ---")
        print(f"Tổng số dòng: {stats['total_rows']}")
        print(f"Hợp lệ: {stats['valid_rows']}")
        print(f"Loại bỏ: {stats['banned_rows']}")
        print(f"Master Items: {len(unique_products)}")

        # BƯỚC 4: GHI FILE (UTF-8-SIG)
        master_fieldnames = ['itemid', 'item_name', 'raw_name', 'current_price', 'total_sold', 'last_updated', 'status']
        with open(OUTPUT_MASTER, mode='w', encoding='utf-8-sig', newline='') as f_master:
            writer = csv.DictWriter(f_master, fieldnames=master_fieldnames)
            writer.writeheader()
            writer.writerows(unique_products.values())
        print(f"\n-> Master created: {OUTPUT_MASTER}")
            
        with open(OUTPUT_HISTORY, mode='w', encoding='utf-8-sig', newline='') as f_history:
            writer = csv.DictWriter(f_history, fieldnames=history_headers)
            writer.writeheader()
            writer.writerows(history_rows)
        print(f"-> History created (Optimized): {OUTPUT_HISTORY}")
        
    except Exception as e:
        print(f"Lỗi: {e}")

# Chạy hàm xử lý
process_shopee_data()

Đang đọc dữ liệu từ: D:\GREENMIND\data_vn_ecommerce\Shopee_Products_Master.csv ...

--- KẾT QUẢ XỬ LÝ (TỐI ƯU V3) ---
Tổng số dòng: 38432
Hợp lệ: 36280
Loại bỏ: 2152
Master Items: 10

-> Master created: D:\GREENMIND\data_vn_ecommerce\Optimized\Core_Product_Master.csv
-> History created (Optimized): D:\GREENMIND\data_vn_ecommerce\Optimized\Price_Stock_History.csv
